# 01 - Dataset Exploration (IO-VNBD, Phase 1)

This notebook explores the raw IO-VNBD dataset used in the Intelligent Dead Reckoning pipeline. It documents the physical layout, trip inventory, canonical schema, sampling behaviour, missing values and GNSS coverage of the smartphone streams.

Everything is driven by the Phase 1 data modules in `src/data/`.

In [ ]:
import sys, os
from pathlib import Path
for p in ("..", "."):
    if (Path(p) / "src").is_dir():
        sys.path.insert(0, os.path.abspath(p)); break

## 1. Setup - point at the repo root

In [ ]:
from src.data.io_vnbd_loader import IOVNBDDataset, detect_sync_status
from src.data.smartphone_extractor import SmartphoneExtractor

ds = IOVNBDDataset("data/raw")
print("raw root:", ds.raw_root)

## 2. Trip inventory

In [ ]:
smart = ds.list_smartphone_files()
veh = ds.list_vehicle_files()
print(f"smartphone files: {len(smart)}")
print(f"vehicle files   : {len(veh)}")
print(f"unique trips    : {len(ds.smartphone_trip_ids())}")

In [ ]:
from collections import Counter
a = Counter(detect_sync_status(p).value for p in smart)
b = Counter(detect_sync_status(p).value for p in veh)
print("smartphone by status:", dict(a))
print("vehicle by status   :", dict(b))

## 3. Canonical smartphone schema

In [ ]:
ex = SmartphoneExtractor(ds)
rows = sorted(ds.smartphone_trip_ids())[:8]
td = ex.extract_trip(rows[0])
d = td.data
print("trip:", td.trip_id)
print("metadata:", td.metadata.to_dict())
print("columns:", list(d.columns))

In [ ]:
print("first rows:")
print(d.head(3).iloc[:, :8].to_string())
print("\nmissing values per column:")
print(d.isna().sum().to_string())

## 4. Sampling behaviour

In [ ]:
dt = d["timestamp"].diff().dropna()
print("median inter-sample delta (s):", round(float(dt.median()), 4))
print("mean   inter-sample delta (s):", round(float(dt.mean()), 4))
print("nominal rate estimate (Hz)   :", round(1 / float(dt.median()), 2))

In [ ]:
gps = d["latitude_deg"].notna()
print(f"GNSS fix coverage: {gps.mean()*100:.2f}% of rows")
print("\ncoverage by trip (first 10):")
for tid in sorted(ds.smartphone_trip_ids())[:10]:
    c = ex.extract_trip(tid).data["latitude_deg"].notna().mean()
    print(f"  {tid:8s} {c*100:5.1f}%")

## 5. Trip metadata summary

In [ ]:
import pandas as pd
metas = [ex.extract_trip(tid).metadata.to_dict() for tid in ds.smartphone_trip_ids()]
meta = pd.DataFrame(metas)
print("drivers:", meta["driver"].value_counts().to_dict())
print("categories:", meta["category"].value_counts().to_dict())
print("sync:", meta["sync_status"].value_counts().to_dict())
print("n_samples range:", meta["n_samples"].min(), "-", meta["n_samples"].max())

## 6. Key findings

- 216 smartphone files / 72 unique trips; 166 vehicle files.
- Same 72 trip ids appear in both **synchronised** and **unsynchronised** folders (different recordings).
- Smartphone streams: 24 columns, 10 Hz nominal, ~1-10% of rows carry a fresh GNSS fix.
- No missing values in the raw CSV body; timestamps monotonic.
- Vehicle files are reference-only (never runtime inputs).

See `docs/dataset.md` for the full write-up.